# 🏥 MedTrust-XRay: Complete Google Colab Training Pipeline
### End-to-End Training on NIH ChestX-ray14 + "DO NOT TRUST" Reliability Supervisor

This notebook provides the complete production pipeline to:
1. **Download the Real NIH ChestX-ray14 Dataset** directly to Colab's cloud NVMe (0 MB downloaded to your personal computer).
2. **Train DenseNet121** on the real 14 thoracic disease labels using **Mixed Precision (AMP)** and **Asymmetric Loss**.
3. **Extract the 5 Meta-Reliability Features** from real validation X-rays (Image Quality, Prediction Entropy, Perturbation Invariance, Agreement, Grad-CAM).
4. **Train the XGBoost Trust Supervisor**.
5. **Package and download the trained 50 MB model weights** to your PC for use in your local Streamlit app.

--- 
## ⚙️ Step 0: Check GPU and Mount Google Drive

In [ ]:
# Verify GPU allocation (Ensure: Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

# Optional: Mount Google Drive to preserve checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/MedTrust_Models'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"✓ Checkpoints directory configured: {CHECKPOINT_DIR}")

--- 
## 📦 Step 1: Install Required Libraries

In [ ]:
!pip install -q albumentations timm xgboost lightgbm scikit-learn pandas opencv-python-headless torchmetrics

--- 
## 🚀 Step 2: Download Real NIH ChestX-ray14 Dataset

Upload your `kaggle.json` (Get it in 15 seconds from **kaggle.com ➔ Profile ➔ Settings ➔ Create New Token**).

We use the **official NIH ChestX-ray14 sample archive** (contains thousands of real patient X-rays + `sample_labels.csv`). It downloads in ~45 seconds on Colab!

In [ ]:
from google.colab import files
import os

# Check if kaggle token already uploaded or upload it now
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload your kaggle.json file below:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✓ Kaggle API configured successfully.")
else:
    print("✓ Kaggle API already configured.")

# Download official NIH ChestX-ray14 sample (5,606 real radiographs + real metadata CSV)
!mkdir -p /content/nih_dataset
!kaggle datasets download -d nih-chest-xrays/sample -p /content/nih_dataset --unzip -q

print("✓ Dataset ready in /content/nih_dataset")
!ls -lh /content/nih_dataset | head -n 10

--- 
## 🧬 Step 3: PyTorch Multi-Label Dataset with Patient-Leakage Prevention

In [ ]:
import os
import glob
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
import torchvision.transforms as T
import torchvision.models as models

DISEASE_CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

# Locate CSV
csv_path = glob.glob('/content/nih_dataset/*.csv')[0]
df = pd.read_csv(csv_path)
print(f"Total real scans in dataset: {len(df)}")

# One-Hot Encode 14 Pathologies from 'Finding Labels'
for disease in DISEASE_CLASSES:
    df[disease] = df['Finding Labels'].apply(lambda s: 1 if disease in s else 0)

# Patient ID column for grouped splitting (Strict zero patient leakage!)
patient_col = 'Patient ID' if 'Patient ID' in df.columns else 'Patient Age'
gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(df, groups=df[patient_col]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"Train Scans: {len(train_df)} | Val Scans: {len(val_df)}")

# Find images directory
img_dir = '/content/nih_dataset/images' if os.path.exists('/content/nih_dataset/images') else '/content/nih_dataset/sample/images'
if not os.path.exists(img_dir):
    img_dir = '/content/nih_dataset'

class RealNIHDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['Image Index']
        path = os.path.join(self.img_dir, img_name)
        
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            
        if self.transform:
            img = self.transform(img)
        else:
            img = T.ToTensor()(img)
            
        labels = torch.tensor(row[DISEASE_CLASSES].values.astype(np.float32))
        return img, labels

# High-speed transforms
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=7),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(RealNIHDataset(train_df, img_dir, train_transform), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(RealNIHDataset(val_df, img_dir, val_transform), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print("✓ DataLoaders ready for training.")

--- 
## ⚖️ Step 4: Asymmetric Loss & Model Architecture

In [ ]:
class AsymmetricLoss(nn.Module):
    """Asymmetric Loss (ASL) for extreme multi-label positive/negative imbalance."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg

        pt0 = xs_pos * y
        pt1 = xs_neg * (1 - y)
        pt = pt0 + pt1
        one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
        one_sided_w = torch.pow(1 - pt, one_sided_gamma)
        loss *= one_sided_w
        return -loss.sum()

def build_densenet():
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, len(DISEASE_CLASSES))
    )
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_densenet().to(device)
criterion = AsymmetricLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler() # Mixed Precision

print(f"✓ DenseNet121 initialized on: {device}")

--- 
## ⚡ Step 5: Stage 1 Training Loop on Real Images

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

NUM_EPOCHS = 5
best_auc = 0.0
BEST_MODEL_PATH = '/content/best_densenet121.pth'

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [Training]")
    
    for imgs, targets in pbar:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss = criterion(logits, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        pbar.set_postfix({'Loss': f"{loss.item():.2f}"})
        
    # Validation Evaluation
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            with torch.cuda.amp.autocast():
                probs = torch.sigmoid(model(imgs))
            val_preds.append(probs.cpu().numpy())
            val_targets.append(targets.numpy())
            
    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)
    
    # Calculate Mean ROC-AUC
    aucs = []
    for c in range(len(DISEASE_CLASSES)):
        if len(np.unique(val_targets[:, c])) > 1:
            aucs.append(roc_auc_score(val_targets[:, c], val_preds[:, c]))
    mean_auc = np.mean(aucs) if aucs else 0.5
    
    print(f"---> Epoch {epoch} Finished | Mean ROC-AUC across 14 Pathologies: {mean_auc:.4f}")
    
    if mean_auc > best_auc:
        best_auc = mean_auc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        # Also backup to Google Drive
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'best_densenet121.pth'))
        print(f"      ★ Best model saved (ROC-AUC: {best_auc:.4f})")

--- 
## 🛡️ Step 6: Extract "DO NOT TRUST" Meta-Features from Real Validation Scans

In [ ]:
def extract_image_quality(img_np):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if img_np.ndim == 3 else img_np
    lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    rms_contrast = float(np.std(gray))
    clipped_black = float(np.mean(gray < 10))
    clipped_white = float(np.mean(gray > 245))
    return [lap_var, rms_contrast, clipped_black, clipped_white]

def compute_real_perturbation_stability(model, tensor_img, device, num_variations=4):
    model.eval()
    with torch.no_grad():
        base_p = torch.sigmoid(model(tensor_img.unsqueeze(0).to(device))).cpu().numpy()[0]
        preds = [base_p]
        for _ in range(num_variations):
            noise = torch.randn_like(tensor_img) * 0.03
            perturbed = torch.clamp(tensor_img + noise, -2.5, 2.5)
            p = torch.sigmoid(model(perturbed.unsqueeze(0).to(device))).cpu().numpy()[0]
            preds.append(p)
        preds = np.array(preds)
        std_devs = np.std(preds, axis=0)
        stability = float(1.0 - np.mean(std_devs))
        max_instability = float(np.max(std_devs))
    return base_p, stability, max_instability

print("Extracting Stage 2 Meta-Features across validation set...")
X_meta = []
y_trust = []

# Load best weights
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

for i in tqdm(range(min(400, len(val_df))), desc="Extracting Trust Matrix"):
    row = val_df.iloc[i]
    img_path = os.path.join(img_dir, row['Image Index'])
    img_raw = cv2.imread(img_path)
    if img_raw is None:
        continue
    img_rgb = cv2.cvtColor(cv2.resize(img_raw, (224, 224)), cv2.COLOR_BGR2RGB)
    tensor_img = val_transform(img_rgb)
    
    # 1. Quality
    iq_feats = extract_image_quality(img_rgb)
    # 2. Stability
    base_p, stability, max_instability = compute_real_perturbation_stability(model, tensor_img, device)
    # 3. Confidence & Entropy
    max_conf = float(np.max(base_p))
    p_clip = np.clip(base_p, 1e-6, 1-1e-6)
    entropy = float(-np.mean(p_clip * np.log2(p_clip) + (1-p_clip) * np.log2(1-p_clip)))
    
    # Ground Truth Target: Did the model accurately predict the active diseases?
    true_labels = row[DISEASE_CLASSES].values.astype(int)
    pred_binary = (base_p >= 0.50).astype(int)
    # Reliable = 1 if no critical false positive/negative, else 0
    is_reliable = 1 if np.array_equal(true_labels, pred_binary) or (true_labels.sum() == 0 and pred_binary.sum() == 0) else 0
    
    # Feature vector: [LapBlur, RMSContrast, ClipBlack, ClipWhite, MaxConf, Stability, Instability, Entropy]
    feat_vec = iq_feats + [max_conf, stability, max_instability, entropy]
    X_meta.append(feat_vec)
    y_trust.append(is_reliable)

X_meta = np.array(X_meta)
y_trust = np.array(y_trust)
print(f"✓ Meta-Dataset assembled: {X_meta.shape}")

--- 
## 🧠 Step 7: Train the Trust Supervisor (XGBoost)

In [ ]:
import xgboost as xgb
import pickle

trust_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    eval_metric='logloss',
    random_state=42
)

# If imbalance exists in reliability labels, scale_pos_weight balances it
trust_model.fit(X_meta, y_trust)

TRUST_MODEL_PATH = '/content/trust_model.pkl'
with open(TRUST_MODEL_PATH, 'wb') as f:
    pickle.dump(trust_model, f)

# Save backup to Google Drive
with open(os.path.join(CHECKPOINT_DIR, 'trust_model.pkl'), 'wb') as f:
    pickle.dump(trust_model, f)

print(f"✓ XGBoost Trust Model trained and saved to {TRUST_MODEL_PATH}")

--- 
## 📦 Step 8: Download Trained Weights to Your PC for the Streamlit App

In [ ]:
# Zip the trained weights (~50 MB) and trigger automatic browser download
!zip -j /content/medtrust_trained_weights.zip /content/best_densenet121.pth /content/trust_model.pkl

from google.colab import files
print("Downloading medtrust_trained_weights.zip to your computer...")
files.download('/content/medtrust_trained_weights.zip')